# Projet Big Data — Analyse Steam avec PySpark

## Contexte

Vous travaillez pour Ubisoft et vous devez analyser le catalogue de jeux vidéo disponible sur Steam afin de mieux comprendre l'écosystème du jeu vidéo.

## Objectifs

Cette analyse exploratoire vise à répondre à plusieurs questions :

- Quels éditeurs publient le plus de jeux ?
- Quels jeux sont les mieux notés ?
- Comment évolue le nombre de sorties dans le temps ?
- Comment les prix sont-ils distribués ?
- Quels genres sont les plus représentés ?
- Quels genres semblent les plus appréciés ?
- Quels genres semblent les plus lucratifs ?
- Les jeux sont-ils majoritairement disponibles sur Windows, Mac ou Linux ?
- Quelles langues sont les plus représentées ?
- Quelle est la répartition des restrictions d'âge ?

> Notebook conçu pour Databricks Free / Community Edition avec PySpark et `display()` pour les visualisations Databricks.


# 1. Chargement des données

Le dataset est fourni sur S3 :

`s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json`

Sur Databricks Free, l'accès direct à S3 peut parfois ne pas fonctionner selon la configuration du cluster.  
La cellule suivante essaie donc plusieurs méthodes de chargement.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when, explode_outer
from pyspark.sql.types import StructType, ArrayType, StringType, IntegerType, DoubleType, BooleanType

spark

In [0]:
# Option 1 : chargement direct depuis S3
s3_path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

try:
    df = spark.read.json(s3_path)
    print("Chargement réussi depuis S3")
except Exception as e:
    print("Le chargement direct S3 a échoué.")
    print("Erreur :", e)
    df = None

Chargement réussi depuis S3


## Alternative si le chargement S3 échoue

Si la cellule précédente affiche une erreur, lancez la cellule suivante pour télécharger le fichier depuis l'URL publique du bucket S3.

> Sur Databricks, les cellules commençant par `%sh` exécutent une commande shell sur le driver.


In [0]:
# À exécuter uniquement si df est None

if df is None:
    import urllib.request
    import os

    urls = [
        "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json",
        "https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json"
    ]

    local_path = "/databricks/driver/steam_game_output.json"

    last_error = None
    for url in urls:
        try:
            print(f"Tentative de téléchargement : {url}")
            urllib.request.urlretrieve(url, local_path)
            print("Téléchargement réussi :", local_path)
            break
        except Exception as e:
            last_error = e
            print("Échec avec cette URL :", e)

    if not os.path.exists(local_path):
        raise Exception(f"Impossible de télécharger le fichier. Dernière erreur : {last_error}")

    df = spark.read.json("file:/databricks/driver/steam_game_output.json")
    print("Chargement réussi depuis le fichier local")

In [0]:
display(df.limit(10))

data,id
"List(10, List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP), 13990, Valve, 0, Action, https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513, 999, English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean, Counter-Strike, 5199, 10,000,000 .. 20,000,000, List(true, true, true), 201215, 999, Valve, 2000/11/1, 0, Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role., List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192), game, )",10
"List(1000000, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud), 0, IndigoBlue Game Studio, 0, Action, Adventure, Indie, https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048, 999, English, Korean, Simplified Chinese, ASCENXION, 5, 0 .. 20,000, List(false, false, true), 27, 999, PsychoFlux Entertainment, 2021/05/14, 0, ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, 

In [0]:
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

# 2. Fonctions utilitaires

Le fichier Steam est semi-structuré : certaines colonnes peuvent être imbriquées.  
Ces fonctions rendent le notebook plus robuste si le schéma varie légèrement.


In [0]:
def has_field(schema, path):
    """Vérifie si un chemin de colonne imbriqué existe dans un schéma Spark."""
    parts = path.split(".")
    current = schema

    for part in parts:
        if isinstance(current, StructType):
            field_names = [field.name for field in current.fields]
            if part not in field_names:
                return False
            current = current[part].dataType
        elif isinstance(current, ArrayType) and isinstance(current.elementType, StructType):
            current = current.elementType
            field_names = [field.name for field in current.fields]
            if part not in field_names:
                return False
            current = current[part].dataType
        else:
            return False

    return True


def safe_col(df, path, alias=None, default=None):
    """Retourne une colonne si elle existe, sinon une valeur par défaut."""
    if has_field(df.schema, path):
        return col(path).alias(alias or path.replace(".", "_"))
    return lit(default).alias(alias or path.replace(".", "_"))


def first_existing_col(df, paths, alias, default=None):
    """Retourne la première colonne existante parmi plusieurs chemins possibles."""
    for path in paths:
        if has_field(df.schema, path):
            return col(path).alias(alias)
    return lit(default).alias(alias)

# 3. Préparation du dataframe principal

On crée un dataframe propre avec les colonnes nécessaires pour l'analyse.

Remarque : certains datasets Steam contiennent les informations dans une colonne imbriquée `data`.  
La cellule suivante gère ce cas automatiquement.


In [0]:
# Si les informations sont dans une colonne imbriquée "data", on l'aplatit.
if has_field(df.schema, "data"):
    if "appid" in df.columns:
        df_base = df.select(col("appid").alias("appid_source"), col("data.*"))
    else:
        df_base = df.select(col("data.*"))
else:
    df_base = df

print("Colonnes disponibles après premier aplatissement :")
print(df_base.columns)
df_base.printSchema()

Colonnes disponibles après premier aplatissement :
['appid', 'categories', 'ccu', 'developer', 'discount', 'genre', 'header_image', 'initialprice', 'languages', 'name', 'negative', 'owners', 'platforms', 'positive', 'price', 'publisher', 'release_date', 'required_age', 'short_description', 'tags', 'type', 'website']
root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable

In [0]:
# Gestion de la date de sortie : parfois string, parfois struct release_date.date
release_date_expr = first_existing_col(
    df_base,
    ["release_date.date", "release_date"],
    "release_date_raw"
)

df_clean = df_base.select(
    first_existing_col(df_base, ["appid", "steam_appid", "appid_source"], "appid"),
    first_existing_col(df_base, ["name"], "name"),
    release_date_expr,

    first_existing_col(df_base, ["price", "initialprice", "price_overview.final"], "price"),
    first_existing_col(df_base, ["discount", "price_overview.discount_percent"], "discount_percent"),
    first_existing_col(df_base, ["is_free"], "is_free"),

    first_existing_col(df_base, ["positive", "recommendations.total"], "positive"),
    first_existing_col(df_base, ["negative"], "negative"),

    first_existing_col(df_base, ["required_age"], "required_age"),

    first_existing_col(df_base, ["publisher", "publishers"], "publishers"),
    first_existing_col(df_base, ["developer", "developers"], "developers"),
    first_existing_col(df_base, ["genre", "genres"], "genres"),
    first_existing_col(df_base, ["categories"], "categories"),

    first_existing_col(df_base, ["platforms.windows"], "windows"),
    first_existing_col(df_base, ["platforms.mac"], "mac"),
    first_existing_col(df_base, ["platforms.linux"], "linux"),

    first_existing_col(df_base, ["languages", "supported_languages"], "supported_languages")
)

display(df_clean.limit(10))

appid,name,release_date_raw,price,discount_percent,is_free,positive,negative,required_age,publishers,developers,genres,categories,windows,mac,linux,supported_languages
10,Counter-Strike,2000/11/1,null,null,null,201215,5199,0,null,null,null,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",true,true,true,null
1000000,ASCENXION,2021/05/14,null,null,null,27,5,0,null,null,null,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",true,false,false,null
1000010,Crown Trick,2020/10/16,null,null,null,4032,646,0,null,null,null,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)",true,false,false,null
1000030,"Cook, Serve, Delicious! 3?!",2020/10/14,null,null,null,1575,115,0,null,null,null,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)",true,true,false,null
1000040,细胞战争,2019/03/30,null,null,null,0,1,0,null,null,null,List(Single-player),true,false,false,null
1000080,Zengeon,2019/06/24,null,null,null,1018,462,0,null,null,null,"List(Multi-player, Single-player, Steam Achievements, Full controller support, Steam Trading Cards)",true,true,false,null
1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,2019/01/24,null,null,null,18,6,0,null,null,null,"List(Single-player, Steam Achievements, Steam Cloud)",true,false,false,null
1000110,Jumping Master(跳跳大咖),2019/04/8,null,null,null,50,34,0,null,null,null,"List(Multi-player, Single-player, Co-op, Online PvP, Online Co-op, PvP)",true,false,false,null
1000130,Cube Defender,2019/01/6,null,null,null,6,0,0,null,null,null,"List(Single-player, Steam Achievements, Steam Leaderboards)",true,true,false,null
1000280,Tower of Origin2-Worm's Nest,2021/09/9,null,null,null,32,12,0,null,null,null,List(Single-player),true,false,false,null


In [0]:
# Nettoyage et cration de variables d'analyse

# 1. Nettoyage de la date brute
df_clean = df_clean.withColumn(
    "release_date_raw_clean",
    F.trim(F.col("release_date_raw").cast("string"))
)

# Remplace les "/" par "-" (ex: "2019/01/15" -> "2019-01-15")
df_clean = df_clean.withColumn(
    "release_date_raw_clean",
    F.regexp_replace(F.col("release_date_raw_clean"), "/", "-")
)

# Met  NULL les chanes vides ou non renseignes AVANT toute normalisation
df_clean = df_clean.withColumn(
    "release_date_raw_clean",
    F.when(
        F.col("release_date_raw_clean").isNull()
        | (F.trim(F.col("release_date_raw_clean")) == "")
        | F.lower(F.col("release_date_raw_clean")).isin("coming soon", "tbd", "to be announced"),
        F.lit(None)
    ).otherwise(F.col("release_date_raw_clean"))
)

# 2. Normalisation des formats incomplets
df_clean = df_clean.withColumn(
    "release_date_normalized",
    F.when(
        F.col("release_date_raw_clean").isNull(),
        F.lit(None)
    ).when(
        F.col("release_date_raw_clean").rlike(r"^\d{4}$"),
        F.concat(F.col("release_date_raw_clean"), F.lit("-01-01"))
    ).when(
        F.col("release_date_raw_clean").rlike(r"^\d{4}-\d{1,2}$"),
        F.concat(F.col("release_date_raw_clean"), F.lit("-01"))
    ).otherwise(F.col("release_date_raw_clean"))
)

# 3. Parsing de la date avec plusieurs formats
df_clean = df_clean.withColumn(
    "release_date",
    F.coalesce(
        F.expr("try_to_date(release_date_normalized, 'yyyy-MM-dd')"),
        F.expr("try_to_date(release_date_normalized, 'yyyy-M-d')"),
        F.expr("try_to_date(release_date_normalized, 'MMM d, yyyy')"),
        F.expr("try_to_date(release_date_normalized, 'MMMM d, yyyy')"),
        F.expr("try_to_date(release_date_normalized, 'd MMM, yyyy')"),
        F.expr("try_to_date(release_date_normalized, 'd MMMM, yyyy')")
    )
)

df_clean = df_clean.withColumn("release_year", F.year(F.col("release_date")))

# 4. Prix (exprim en centimes dans le dataset Steam, ex: '999' -> 9.99)
df_clean = df_clean.withColumn(
    "price_euros",
    F.coalesce(
        F.round(F.regexp_replace(F.col("price").cast("string"), r"[^0-9.]", "").cast("double") / 100, 2),
        F.lit(0.0)
    )
)

# 4bis. Dtection fiable des jeux gratuits (is_free)
df_clean = df_clean.withColumn(
    "is_free",
    F.when(
        (F.col("price_euros") == 0) | (F.col("is_free") == True) | (F.col("price").cast("string") == "0"),
        F.lit(True)
    ).otherwise(F.lit(False))
)

# 5. Avis et ratios
df_clean = df_clean.withColumn(
    "positive",
    F.coalesce(F.col("positive").cast("double"), F.lit(0.0))
).withColumn(
    "negative",
    F.coalesce(F.col("negative").cast("double"), F.lit(0.0))
).withColumn(
    "total_reviews",
    F.col("positive") + F.col("negative")
).withColumn(
    "positive_ratio",
    F.when(F.col("total_reviews") > 0, F.col("positive") / F.col("total_reviews")).otherwise(F.lit(0.0))
)

# 6. Remise (pourcentage de rduction, ex: '70' -> 70.0)
df_clean = df_clean.withColumn(
    "discount_percent",
    F.coalesce(F.regexp_replace(F.col("discount_percent").cast("string"), r"[^0-9.]", "").cast("double"), F.lit(0.0))
)

display(df_clean.limit(10))

appid,name,release_date_raw,price,discount_percent,is_free,positive,negative,required_age,publishers,developers,genres,categories,windows,mac,linux,supported_languages,release_date_raw_clean,release_date_normalized,release_date,release_year,price_euros,total_reviews,positive_ratio
10,Counter-Strike,2000/11/1,null,0.0,null,201215.0,5199.0,0,null,null,null,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",true,true,true,null,2000-11-1,2000-11-1,2000-11-01,2000,null,206414.0,0.9748127549487923
1000000,ASCENXION,2021/05/14,null,0.0,null,27.0,5.0,0,null,null,null,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",true,false,false,null,2021-05-14,2021-05-14,2021-05-14,2021,null,32.0,0.84375
1000010,Crown Trick,2020/10/16,null,0.0,null,4032.0,646.0,0,null,null,null,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)",true,false,false,null,2020-10-16,2020-10-16,2020-10-16,2020,null,4678.0,0.8619067977768277
1000030,"Cook, Serve, Delicious! 3?!",2020/10/14,null,0.0,null,1575.0,115.0,0,null,null,null,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)",true,true,false,null,2020-10-14,2020-10-14,2020-10-14,2020,null,1690.0,0.9319526627218935
1000040,细胞战争,2019/03/30,null,0.0,null,0.0,1.0,0,null,null,null,List(Single-player),true,false,false,null,2019-03-30,2019-03-30,2019-03-30,2019,null,1.0,0.0
1000080,Zengeon,2019/06/24,null,0.0,null,1018.0,462.0,0,null,null,null,"List(Multi-player, Single-player, Steam Achievements, Full controller support, Steam Trading Cards)",true,true,false,null,2019-06-24,2019-06-24,2019-06-24,2019,null,1480.0,0.6878378378378378
1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,2019/01/24,null,0.0,null,18.0,6.0,0,null,null,null,"List(Single-player, Steam Achievements, Steam Cloud)",true,false,false,null,2019-01-24,2019-01-24,2019-01-24,2019,null,24.0,0.75
1000110,Jumping Master(跳跳大咖),2019/04/8,null,0.0,null,50.0,34.0,0,null,null,null,"List(Multi-player, Single-player, Co-op, Online PvP, Online Co-op, PvP)",true,false,false,null,2019-04-8,2019-04-8,2019-04-08,2019,null,84.0,0.5952380952380952
1000130,Cube Defender,2019/01/6,null,0.0,null,6.0,0.0,0,null,null,null,"List(Single-player, Steam Achievements, Steam Leaderboards)",true,true,false,null,2019-01-6,2019-01-6,2019-01-06,2019,null,6.0,1.0
1000280,Tower of Origin2-Worm's Nest,2021/09/9,null,0.0,null,32.0,12.0,0,null,null,null,List(Single-player),true,false,false,null,2021-09-9,2021-09-9,2021-09-09,2021,null,44.0,0.7272727272727273


In [0]:
df_clean.select(
    "appid", "name", "release_date", "release_year", "price_euros",
    "discount_percent", "positive", "negative", "total_reviews",
    "positive_ratio", "required_age", "windows", "mac", "linux"
).summary().show()

+-------+------------------+------------------------+------------------+-----------+----------------+------------------+------------------+------------------+-------------------+------------------+
|summary|             appid|                    name|      release_year|price_euros|discount_percent|          positive|          negative|     total_reviews|     positive_ratio|      required_age|
+-------+------------------+------------------------+------------------+-----------+----------------+------------------+------------------+------------------+-------------------+------------------+
|  count|             55691|                   55691|             55592|          0|           55691|             55691|             55691|             55691|              55528|             55691|
|   mean|1025603.0926720655|                Infinity|2018.6526298748022|       NULL|             0.0|1470.8755992889335| 241.8376937027527|1712.7132929916863| 0.7365355475592407|0.1978882344490734|
| stddev| 

# 4. Analyse macro

## 4.1 Nombre total de jeux


In [0]:
total_games = df_clean.select("appid").distinct().count()
print(f"Nombre total de jeux dans le dataset : {total_games}")

Nombre total de jeux dans le dataset : 55691


## 4.2 Nombre de sorties par année

Visualisation recommandée dans Databricks :

- Type : Line chart ou Bar chart
- X : `release_year`
- Y : `count`


In [0]:
games_per_year = df_clean.filter(col("release_year").isNotNull()) \
    .groupBy("release_year") \
    .count() \
    .orderBy("release_year")

display(games_per_year)

release_year,count
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


### Analyse Covid : avant, pendant, après

On compare trois périodes :

- Avant Covid : jusqu'à 2019
- Covid : 2020-2021
- Après Covid : à partir de 2022


In [0]:
covid_periods = df_clean.filter(col("release_year").isNotNull()) \
    .withColumn(
        "period",
        when(col("release_year") <= 2019, "Avant Covid")
        .when((col("release_year") >= 2020) & (col("release_year") <= 2021), "Covid")
        .otherwise("Après Covid")
    ) \
    .groupBy("period") \
    .count() \
    .orderBy("period")

display(covid_periods)

period,count
Après Covid,7455
Avant Covid,31009
Covid,17128


## 4.3 Éditeurs ayant publié le plus de jeux

Comme un jeu peut avoir plusieurs éditeurs, on utilise `explode_outer()`.


In [0]:
from pyspark.sql.types import ArrayType

# Dtection automatique du format des diteurs (tableau ou chane dlimite par virgule)
publishers_type = df_clean.schema["publishers"].dataType

if isinstance(publishers_type, ArrayType):
    publishers_df = df_clean.select(
        "appid",
        "name",
        F.explode_outer("publishers").alias("publisher")
    ).filter(F.col("publisher").isNotNull() & (F.trim(F.col("publisher")) != ""))
else:
    publishers_df = df_clean.select(
        "appid",
        "name",
        F.explode_outer(F.split(F.col("publishers").cast("string"), r",\s*")).alias("publisher")
    ).filter(F.col("publisher").isNotNull() & (F.trim(F.col("publisher")) != ""))

top_publishers = publishers_df.groupBy("publisher") \
    .count() \
    .orderBy(F.col("count").desc())

display(top_publishers.limit(20))

root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date_raw: string (nullable = true)
 |-- price: void (nullable = true)
 |-- discount_percent: double (nullable = false)
 |-- is_free: void (nullable = true)
 |-- positive: double (nullable = true)
 |-- negative: double (nullable = true)
 |-- required_age: string (nullable = true)
 |-- publishers: void (nullable = true)
 |-- developers: void (nullable = true)
 |-- genres: void (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- supported_languages: void (nullable = true)
 |-- release_date_raw_clean: string (nullable = true)
 |-- release_date_normalized: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- price_euros: double (nullable = true)
 |-- total_reviews: doub

publisher,count


In [0]:
for c in df_base.columns:
    if "publish" in c.lower():
        print(c)

publisher


In [0]:
display(df_base.select([c for c in df_base.columns if "publish" in c.lower()]).limit(10))

publisher
Valve
PsychoFlux Entertainment
"Team17, NEXT Studios"
Vertigo Gaming Inc.
DoubleC Games
2P Games
Starship Studio
重庆环游者网络科技
Simon Codrington
Villain Role


In [0]:
for c in df_base.columns:
    if "genre" in c.lower():
        print(c)

genre


## 4.4 Jeux les mieux notés

On classe les jeux selon leur ratio d'avis positifs.

Pour éviter un biais, on garde uniquement les jeux ayant au moins 500 avis.


In [0]:
best_rated_games = df_clean.filter(col("total_reviews") >= 500) \
    .orderBy(col("positive_ratio").desc())

display(
    best_rated_games.select(
        "name",
        "positive",
        "negative",
        "total_reviews",
        F.round("positive_ratio", 4).alias("positive_ratio")
    ).limit(20)
)

name,positive,negative,total_reviews,positive_ratio
Flowers -Le volume sur ete-,937.0,1.0,938.0,0.9989
Touhou Fuujinroku ~ Mountain of Faith.,649.0,4.0,653.0,0.9939
Aventura Copilului Albastru și Urât,2203.0,14.0,2217.0,0.9937
The Case of the Golden Idol,909.0,6.0,915.0,0.9934
Aseprite,11823.0,80.0,11903.0,0.9933
Patrick's Parabox,1489.0,11.0,1500.0,0.9927
A Short Hike,11645.0,87.0,11732.0,0.9926
Monster Prom 3: Monster Roadtrip,1052.0,8.0,1060.0,0.9925
Flowers -Le volume sur automne-,516.0,4.0,520.0,0.9923
CULTIC,2021.0,16.0,2037.0,0.9921


# 5. Analyse des prix

## 5.1 Distribution des prix

Visualisation recommandée :

- Histogram
- Valeur : `price_euros`


In [0]:
price_distribution = df_clean.select(
    "name",
    "price_euros",
    "discount_percent",
    "is_free"
).filter(col("price_euros").isNotNull())

display(price_distribution)

name,price_euros,discount_percent,is_free


In [0]:
df_clean.select("price_euros", "discount_percent").describe().show()

+-------+-----------+----------------+
|summary|price_euros|discount_percent|
+-------+-----------+----------------+
|  count|          0|           55691|
|   mean|       NULL|             0.0|
| stddev|       NULL|             0.0|
|    min|       NULL|             0.0|
|    max|       NULL|             0.0|
+-------+-----------+----------------+



## 5.2 Jeux gratuits vs payants


In [0]:
free_paid = df_clean.withColumn(
    "price_type",
    when((col("price_euros") == 0) | (col("is_free") == True), "Gratuit")
    .otherwise("Payant")
).groupBy("price_type").count()

display(free_paid)

price_type,count
Payant,55691


## 5.3 Jeux avec réduction


In [0]:
discount_analysis = df_clean.withColumn(
    "discount_status",
    when(col("discount_percent") > 0, "Avec réduction")
    .otherwise("Sans réduction")
).groupBy("discount_status").count()

display(discount_analysis)

discount_status,count
Sans réduction,55691


In [0]:
top_discounts = df_clean.filter(col("discount_percent") > 0) \
    .select("name", "price_euros", "discount_percent", "positive_ratio", "total_reviews") \
    .orderBy(col("discount_percent").desc())

display(top_discounts.limit(30))

name,price_euros,discount_percent,positive_ratio,total_reviews


# 6. Analyse des genres

Les genres sont stockés sous forme de liste de structures.  
On les transforme en une ligne par jeu et par genre.


In [0]:
genres_df_with_genre = df_base.select(
    first_existing_col(df_base, ["appid", "steam_appid", "appid_source"], "appid"),
    first_existing_col(df_base, ["name"], "name"),
    first_existing_col(df_base, ["genre", "genres"], "genre")
).filter(col("genre").isNotNull())

# Séparation des genres multiples (ex: "Action, Adventure, Indie" -> 1 ligne par genre)
genres_df_with_genre = genres_df_with_genre.withColumn(
    "genre",
    F.explode_outer(F.split(F.col("genre").cast("string"), r",\s*"))
).filter((col("genre").isNotNull()) & (F.trim(col("genre")) != ""))

# Jointure avec df_clean pour avoir les avis ainsi que les prix
genres_df = genres_df_with_genre.join(
    df_clean.select("appid", "positive", "negative", "total_reviews", "positive_ratio", "price_euros"),
    on="appid",
    how="left"
)

display(genres_df.limit(20))

appid,name,genre,positive,negative,total_reviews,positive_ratio,price_euros
10,Counter-Strike,Action,201215.0,5199.0,206414.0,0.9748127549487923,null
1000000,ASCENXION,"Action, Adventure, Indie",27.0,5.0,32.0,0.84375,null
1000010,Crown Trick,"Adventure, Indie, RPG, Strategy",4032.0,646.0,4678.0,0.8619067977768277,null
1000030,"Cook, Serve, Delicious! 3?!","Action, Indie, Simulation, Strategy",1575.0,115.0,1690.0,0.9319526627218935,null
1000040,细胞战争,"Action, Casual, Indie, Simulation",0.0,1.0,1.0,0.0,null
1000080,Zengeon,"Action, Adventure, Indie, RPG",1018.0,462.0,1480.0,0.6878378378378378,null
1000100,干支セトラ 陽ノ卷｜干支etc. 陽之卷,"Adventure, Indie, RPG, Strategy",18.0,6.0,24.0,0.75,null
1000110,Jumping Master(跳跳大咖),"Action, Adventure, Casual, Free to Play, Massively Multiplayer",50.0,34.0,84.0,0.5952380952380952,null
1000130,Cube Defender,"Casual, Indie",6.0,0.0,6.0,1.0,null
1000280,Tower of Origin2-Worm's Nest,"Indie, RPG",32.0,12.0,44.0,0.7272727272727273,null


## 6.1 Genres les plus représentés


In [0]:
top_genres = genres_df.groupBy("genre") \
    .count() \
    .orderBy(col("count").desc())

display(top_genres)

genre,count
"Action, Indie",3460
"Casual, Indie",3060
"Action, Adventure, Indie",2783
"Adventure, Indie",2316
"Action, Casual, Indie",1914
"Adventure, Casual, Indie",1811
Indie,1756
Action,1633
Casual,1433
Adventure,1021


## 6.2 Genres avec le meilleur ratio d'avis positifs

On garde uniquement les genres ayant au moins 50 jeux pour éviter les conclusions sur des volumes trop faibles.


In [0]:
genre_rating = genres_df.groupBy("genre").agg(
    F.avg("positive_ratio").alias("avg_positive_ratio"),
    F.avg("total_reviews").alias("avg_total_reviews"),
    F.countDistinct("appid").alias("nb_games")
).filter(col("nb_games") >= 50) \
 .orderBy(col("avg_positive_ratio").desc())

display(genre_rating)

genre,avg_positive_ratio,avg_total_reviews,nb_games
"Adventure, Casual, Free to Play, Indie",0.8257282393397695,1299.5727272727272,110
"Action, Casual, Indie, Sports",0.8169714262083794,108.91964285714286,112
"Casual, Free to Play, Indie",0.8080057592007156,2479.4682080924854,173
"Adventure, RPG",0.8033628262224352,841.325,200
"Action, Indie, Sports",0.8028838427533302,387.13157894736844,114
"Casual, Indie, Strategy, Early Access",0.7984672385205065,44.15384615384615,78
"Adventure, Free to Play, Indie",0.7979183189545908,1431.858208955224,134
"Adventure, Casual",0.7969039058593713,174.02586206896552,928
"Adventure, Casual, Indie, RPG, Simulation",0.792733256305468,1155.4,100
"Adventure, Indie, RPG, Strategy",0.7887573787346985,1254.2910798122066,213


## 6.3 Genres les plus lucratifs

Le dataset ne contient pas les ventes exactes.  
On crée donc un proxy de revenus :

`revenue_proxy = prix × nombre total d'avis`

Ce n'est pas un chiffre d'affaires réel, mais un indicateur utile pour comparer les genres.


In [0]:
genre_revenue_proxy = genres_df.groupBy("genre").agg(
    F.sum(col("price_euros") * col("total_reviews")).alias("revenue_proxy"),
    F.avg("price_euros").alias("avg_price"),
    F.avg("positive_ratio").alias("avg_positive_ratio"),
    F.countDistinct("appid").alias("nb_games")
).orderBy(col("revenue_proxy").desc())

display(genre_revenue_proxy)

genre,revenue_proxy,avg_price,avg_positive_ratio,nb_games
"Indie, RPG",null,null,0.773643591523553,444
"Action, Indie, Racing, Simulation, Early Access",null,null,0.888563829787234,4
"Action, Adventure, Indie, Racing, Early Access",null,null,0.9166666666666666,4
"Action, Adventure, Casual, Indie, Simulation, Early Access",null,null,0.7980026522003941,24
"Action, Adventure, Strategy",null,null,0.640897801750758,64
"Action, Free to Play, Indie, Early Access",null,null,0.6667034245184867,36
"Adventure, Casual, Free to Play, Massively Multiplayer, RPG",null,null,0.6322970682483791,5
"Animation & Modeling, Video Production, Game Development",null,null,0.5,1
"Action, Indie, RPG",null,null,0.7264422735309694,379
"Adventure, Simulation, Strategy, Early Access",null,null,0.6483130382492538,3


# 7. Analyse plateformes

## 7.1 Disponibilité Windows / Mac / Linux


In [0]:
windows_df = df_clean.filter(col("windows") == True).select(lit("Windows").alias("platform"))
mac_df = df_clean.filter(col("mac") == True).select(lit("Mac").alias("platform"))
linux_df = df_clean.filter(col("linux") == True).select(lit("Linux").alias("platform"))

platforms_df = windows_df.union(mac_df).union(linux_df)

platform_counts = platforms_df.groupBy("platform").count().orderBy(col("count").desc())

display(platform_counts)

platform,count
Windows,55676
Mac,12770
Linux,8458


## 7.2 Genres par plateforme


In [0]:
genre_platform_df = genres_df.join(
    df_clean.select("appid", "windows", "mac", "linux"),
    on="appid",
    how="left"
)

genre_platform_summary = genre_platform_df.groupBy("genre").agg(
    F.sum(when(col("windows") == True, 1).otherwise(0)).alias("windows_games"),
    F.sum(when(col("mac") == True, 1).otherwise(0)).alias("mac_games"),
    F.sum(when(col("linux") == True, 1).otherwise(0)).alias("linux_games"),
    F.countDistinct("appid").alias("total_games")
).orderBy(col("total_games").desc())

display(genre_platform_summary)

genre,windows_games,mac_games,linux_games,total_games
"Action, Indie",3460,815,617,3460
"Casual, Indie",3060,739,545,3060
"Action, Adventure, Indie",2783,612,477,2783
"Adventure, Indie",2315,770,494,2316
"Action, Casual, Indie",1914,452,351,1914
"Adventure, Casual, Indie",1811,584,372,1811
Indie,1755,431,276,1756
Action,1632,209,155,1633
Casual,1433,294,122,1433
Adventure,1020,232,122,1021


# 8. Analyse des langues

Le champ `supported_languages` est souvent un texte contenant plusieurs langues.  
On détecte les langues principales avec des recherches de chaînes.


In [0]:
languages_df = df_clean.select(
    "appid",
    "name",
    F.lower(F.regexp_replace(col("supported_languages"), "<[^>]+>", "")).alias("languages")
)

language_summary = languages_df.selectExpr(
    "sum(case when languages like '%english%' then 1 else 0 end) as english",
    "sum(case when languages like '%french%' then 1 else 0 end) as french",
    "sum(case when languages like '%german%' then 1 else 0 end) as german",
    "sum(case when languages like '%spanish%' then 1 else 0 end) as spanish",
    "sum(case when languages like '%italian%' then 1 else 0 end) as italian",
    "sum(case when languages like '%japanese%' then 1 else 0 end) as japanese",
    "sum(case when languages like '%korean%' then 1 else 0 end) as korean",
    "sum(case when languages like '%chinese%' then 1 else 0 end) as chinese",
    "sum(case when languages like '%portuguese%' then 1 else 0 end) as portuguese",
    "sum(case when languages like '%russian%' then 1 else 0 end) as russian"
)

display(language_summary)

english,french,german,spanish,italian,japanese,korean,chinese,portuguese,russian
0,0,0,0,0,0,0,0,0,0


Pour obtenir une visualisation plus simple, on transforme le tableau large en format long.


In [0]:
language_long = language_summary.selectExpr(
    "stack(10, " +
    "'English', english, " +
    "'French', french, " +
    "'German', german, " +
    "'Spanish', spanish, " +
    "'Italian', italian, " +
    "'Japanese', japanese, " +
    "'Korean', korean, " +
    "'Chinese', chinese, " +
    "'Portuguese', portuguese, " +
    "'Russian', russian" +
    ") as (language, count)"
).orderBy(col("count").desc())

display(language_long)

language,count
Portuguese,0
Japanese,0
Korean,0
Chinese,0
French,0
Italian,0
English,0
Spanish,0
German,0
Russian,0


# 9. Analyse de l'âge requis

## 9.1 Répartition brute


In [0]:
age_analysis = df_clean.groupBy("required_age") \
    .count() \
    .orderBy("required_age")

display(age_analysis)

required_age,count
0,55030
10,7
12,32
13,26
14,10
15,264
16,38
17,38
18,223
180,4


## 9.2 Catégories d'âge


In [0]:
# Convertion `required_age` en entier, en gérant les valeurs non numériques telles que « MA 15+ », « 21+ », etc.
age_categories = df_clean.withColumn(
    "required_age_int",
    F.expr("try_cast(required_age as int)")
).withColumn(
    "age_category",
    when(col("required_age_int").isNull(), "Non renseigné")
    .when(col("required_age_int") == 0, "Tous publics")
    .when(col("required_age_int") < 16, "Moins de 16 ans")
    .when(col("required_age_int") < 18, "16+")
    .otherwise("18+")
)

age_category_analysis = age_categories.groupBy("age_category").count()

display(age_category_analysis)

age_category,count
Tous publics,55030
Non renseigné,3
18+,229
Moins de 16 ans,353
16+,76


# 10. Analyse éditeurs × genres

Cette analyse permet d'identifier les genres de prédilection des principaux éditeurs.


In [0]:
# Les colonnes réelles sont au singulier (publisher, genre) et non au pluriel, et elles sont de type STRING.
publisher_genre_df = df_base.select(
    first_existing_col(df_base, ["appid", "steam_appid", "appid_source"], "appid"),
    first_existing_col(df_base, ["name"], "name"),
    first_existing_col(df_base, ["publisher"], "publisher"),
    first_existing_col(df_base, ["genre"], "genre")
).filter(
    (col("publisher").isNotNull()) & (col("genre").isNotNull())
)

# Séparer les chaînes séparées par des virgules en tableaux, puis les éclater
publisher_genre_df = publisher_genre_df.withColumn(
    "publisher_array",
    F.split(F.trim(col("publisher")), ",\\s*")
).withColumn(
    "genre_array",
    F.split(F.trim(col("genre")), ",\\s*")
)

# Décomposer les éditeurs
publisher_genre_df = publisher_genre_df.select(
    "appid",
    "name",
    explode_outer("publisher_array").alias("publisher"),
    "genre_array"
)

# Décomposer les genres
publisher_genre_df = publisher_genre_df.select(
    "appid",
    "name",
    "publisher",
    explode_outer("genre_array").alias("genre")
).filter(
    (col("publisher").isNotNull()) & (col("genre").isNotNull()) &
    (F.trim(col("publisher")) != "") & (F.trim(col("genre")) != "")
)

publisher_genre_summary = publisher_genre_df.groupBy("publisher", "genre") \
    .count() \
    .orderBy(col("count").desc())

display(publisher_genre_summary.limit(100))

publisher,genre,count
Big Fish Games,Casual,419
Big Fish Games,Adventure,393
LLC,Indie,321
8floor,Casual,202
LLC,Action,201
Inc.,Action,179
Ltd.,Action,179
Inc.,Indie,164
Inc.,Casual,151
Ltd.,Indie,144


## Focus sur les 20 plus gros éditeurs


In [0]:
# Obtenir les 20 principaux éditeurs selon le nombre total de jeux à partir de publisher_genre_df.
top_20_publishers = publisher_genre_df.groupBy("publisher") \
    .agg(F.countDistinct("appid").alias("total_games")) \
    .orderBy(col("total_games").desc()) \
    .limit(20)

# Combinaison avec publisher_genre_summary pour obtenir la répartition par genre des principaux éditeurs.
publisher_genre_top = publisher_genre_summary.join(
    top_20_publishers.select("publisher"),
    on="publisher",
    how="inner"
).orderBy(col("count").desc())

display(publisher_genre_top)

publisher,genre,count
Big Fish Games,Casual,419
Big Fish Games,Adventure,393
LLC,Indie,321
8floor,Casual,202
LLC,Action,201
Inc.,Action,179
Ltd.,Action,179
Inc.,Indie,164
Inc.,Casual,151
Ltd.,Indie,144


# 11. Analyse de popularité

On étudie les jeux avec le plus grand nombre d'avis.  
Le nombre total d'avis est utilisé comme proxy de popularité.


In [0]:
most_reviewed_games = df_clean.filter(col("total_reviews") > 0) \
    .select("name", "total_reviews", "positive", "negative", "positive_ratio", "price_euros") \
    .orderBy(col("total_reviews").desc())

display(most_reviewed_games.limit(30))

name,total_reviews,positive,negative,positive_ratio,price_euros
Counter-Strike: Global Offensive,6730438.0,5943345.0,787093.0,0.8830547135268165,null
PUBG: BATTLEGROUNDS,2093876.0,1185361.0,908515.0,0.5661084992616564,null
Dota 2,1852811.0,1534895.0,317916.0,0.828414231133127,null
Grand Theft Auto V,1442644.0,1229265.0,213379.0,0.852091714934523,null
Tom Clancy's Rainbow Six Siege,1086157.0,942910.0,143247.0,0.8681157512219688,null
Terraria,1037091.0,1014711.0,22380.0,0.978420408623737,null
Team Fortress 2,903830.0,846407.0,57423.0,0.9364670347299824,null
Garry's Mod,891238.0,861240.0,29998.0,0.9663412017889722,null
Rust,844667.0,732513.0,112154.0,0.8672210468740936,null
Left 4 Dead 2,660664.0,643836.0,16828.0,0.9745286560187932,null


## Popularité moyenne par genre


In [0]:
genre_popularity = genres_df.groupBy("genre").agg(
    F.avg("total_reviews").alias("avg_reviews"),
    F.expr("percentile_approx(total_reviews, 0.5)").alias("median_reviews"),
    F.countDistinct("appid").alias("nb_games")
).filter(col("nb_games") >= 50) \
 .orderBy(col("avg_reviews").desc())

display(genre_popularity)

genre,avg_reviews,median_reviews,nb_games
"Action, Free to Play",99168.13793103448,404.0,87
"Action, RPG",18217.236842105263,571.0,190
"Action, Adventure",9828.02806122449,118.0,784
"Action, Adventure, Casual, Free to Play, Indie",9321.611111111111,53.0,54
"Indie, RPG, Simulation",7411.61038961039,34.0,77
"Action, Adventure, RPG",6347.052401746725,227.0,229
Action,6339.0134721371705,60.0,1633
RPG,6097.619850187266,64.0,534
"Action, Simulation, Strategy",6097.583333333333,56.0,60
"Indie, Simulation",5222.965437788019,39.0,434
